# UPLIFT MODEL SCRIPT (T-LEARNER WITH XGBOOST)
Reads from Excel `.xlsx`  
Outcome: `outcome_ed_90d`  
Treatment: `intervention_flag`


---
## 1. Install packages if needed
---


In [2]:
from pathlib import Path
import importlib.util
import sys

required_imports = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'xgboost': 'xgboost',
    'openpyxl': 'openpyxl',
    'matplotlib': 'matplotlib',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for import_name, pip_name in required_imports.items() if importlib.util.find_spec(import_name) is None]
if missing:
    raise ImportError('Install missing packages with: pip install ' + ' '.join(missing))

for candidate in [Path.cwd(), Path.cwd() / 'Code']:
    if (candidate / '_prism_model_utils.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not find _prism_model_utils.py in the notebook folder or ./Code')


---
## 2. Load packages
---


In [3]:
from itertools import product
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from _prism_model_utils import (
    GITHUB_XLSX_URL,
    align_to_columns,
    assert_xgb_booster_uses_cuda,
    clean_names_simple,
    ensure_output_folder,
    impute_categorical,
    impute_numeric,
    make_design_matrix,
    ntile_desc,
    project_root,
    read_prism_excel,
    require_columns,
    resolve_xgb_gpu_params,
    shap_importance_frame,
    split_train_test,
    to_binary,
    xgb_importance_frame,
    xgb_training_params,
)

warnings.filterwarnings('ignore', category=ConvergenceWarning)
PROJECT_ROOT = project_root()


---
## 3. FILE PATHS
---


In [4]:
# Raw GitHub URL to the Excel file
github_xlsx_url = GITHUB_XLSX_URL

# Output paths: save inside Outputs/Uplift/Python
output_folder = ensure_output_folder(PROJECT_ROOT / 'Outputs' / 'Uplift' / 'Python')
output_path = output_folder / 'uplift_scored_output.csv'
summary_path = output_folder / 'uplift_decile_summary.csv'

print('Imported data from:', github_xlsx_url, '\n')
print('Project root resolved to:', PROJECT_ROOT, '\n')
print('Outputs will be saved to:', output_folder, '\n')


Imported data from: https://raw.githubusercontent.com/ndesai777777/prism_repo/main/DataSets/PRP_1000_full_pretreatment.xlsx 

Project root resolved to: C:\Users\i.Nikesh.Desai\OneDrive - Acentra\Documents\GitHub\prism_repo 

Outputs will be saved to: C:\Users\i.Nikesh.Desai\OneDrive - Acentra\Documents\GitHub\prism_repo\Outputs\Uplift\Python 



---
## 4. HELPER FUNCTIONS
---


In [ ]:
def safe_as_date(values):
    return pd.to_datetime(values, errors='coerce')


def present_columns(columns, df):
    return [column for column in columns if column in df.columns]


def safe_auc(y_true, y_pred, label):
    y_series = pd.Series(y_true).dropna()
    if y_series.nunique() < 2:
        print(f'{label} AUC: cannot calculate because only one outcome class is present')
        return np.nan
    auc_value = roc_auc_score(y_true, y_pred)
    print(f'{label} AUC: {auc_value:.4f}')
    return auc_value


GPU_ONLY_TRAINING = True
RUN_CPU_ONLY_COMPARISON_MODELS = False
XGBOOST_CUDA_DEVICE = 0
XGB_GPU_PARAMS = resolve_xgb_gpu_params(cuda_device=XGBOOST_CUDA_DEVICE) if GPU_ONLY_TRAINING else {}
print('GPU-only training enabled:', GPU_ONLY_TRAINING)
print('XGBoost CUDA params used for training:', XGB_GPU_PARAMS)


def make_dmatrix(x_matrix, y=None):
    if y is None:
        return xgb.DMatrix(x_matrix, feature_names=list(x_matrix.columns))
    return xgb.DMatrix(x_matrix, label=np.asarray(y, dtype=float), feature_names=list(x_matrix.columns))


def fit_xgb_cv_grid(x_matrix, y, grid, nrounds_max=500, nfold=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    dtrain = make_dmatrix(x_matrix, y_array)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfold, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for XGBoost CV.')

    results = []
    best_model_info = None
    best_auc = -np.inf

    for params_grid in grid:
        params_i = xgb_training_params(
            XGB_GPU_PARAMS,
            {
                'max_depth': params_grid['max_depth'],
                'eta': params_grid['eta'],
                'min_child_weight': params_grid['min_child_weight'],
                'subsample': 0.8,
                'colsample_bytree': 0.8,
            },
            eval_metric='auc',
            seed=seed,
        )
        cv_i = xgb.cv(
            params=params_i,
            dtrain=dtrain,
            num_boost_round=nrounds_max,
            nfold=folds,
            stratified=True,
            early_stopping_rounds=20,
            seed=seed,
            verbose_eval=False,
        )
        auc_column = 'test-auc-mean'
        best_iter_i = int(cv_i[auc_column].idxmax())
        best_auc_i = float(cv_i.loc[best_iter_i, auc_column])
        best_nrounds_i = best_iter_i + 1
        results.append({
            'max_depth': params_grid['max_depth'],
            'eta': params_grid['eta'],
            'min_child_weight': params_grid['min_child_weight'],
            'best_nrounds': best_nrounds_i,
            'cv_auc': best_auc_i,
        })
        if best_auc_i > best_auc:
            best_auc = best_auc_i
            best_model_info = {'params': params_i, 'best_nrounds': best_nrounds_i, 'cv_auc': best_auc_i}

    search_results = pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True)
    final_model = xgb.train(
        params=best_model_info['params'],
        dtrain=dtrain,
        num_boost_round=best_model_info['best_nrounds'],
        verbose_eval=False,
    )
    return {
        'model': final_model,
        'best_params': best_model_info['params'],
        'best_nrounds': best_model_info['best_nrounds'],
        'best_cv_auc': best_model_info['cv_auc'],
        'search_results': search_results,
    }


def fit_elastic_net(x_matrix, y, alpha_grid=np.round(np.arange(0, 1.01, 0.1), 1), nfolds=5, seed=123):
    y_array = np.asarray(y, dtype=float)
    class_counts = pd.Series(y_array).value_counts()
    folds = int(min(nfolds, class_counts.min())) if len(class_counts) > 1 else 0
    if folds < 2:
        raise ValueError('Need at least two outcome classes with at least two rows each for elastic-net CV.')

    if GPU_ONLY_TRAINING and not RUN_CPU_ONLY_COMPARISON_MODELS:
        raise RuntimeError(
            'fit_elastic_net uses sklearn LogisticRegressionCV, which trains on CPU. '
            'Keep RUN_CPU_ONLY_COMPARISON_MODELS = False for GPU-only notebook runs, '
            'or set it to True only when you intentionally want this CPU comparison model.'
        )

    results = []
    best_pipeline = None
    best_auc = -np.inf
    best_alpha = np.nan
    best_lambda = np.nan
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    for alpha in alpha_grid:
        penalty = 'l2' if alpha == 0 else 'elasticnet'
        l1_ratios = None if alpha == 0 else [float(alpha)]
        cv_model = LogisticRegressionCV(
            Cs=np.logspace(-4, 4, 30),
            cv=cv,
            penalty=penalty,
            solver='saga',
            l1_ratios=l1_ratios,
            scoring='roc_auc',
            max_iter=10000,
            random_state=seed,
            refit=True,
        )
        pipeline = make_pipeline(StandardScaler(), cv_model)
        pipeline.fit(x_matrix, y_array)
        fitted = pipeline.named_steps['logisticregressioncv']
        scores = fitted.scores_[1.0]
        auc_cv = float(np.nanmax(np.nanmean(scores, axis=0)))
        lambda_value = float(1 / fitted.C_[0])
        results.append({'alpha': float(alpha), 'lambda': lambda_value, 'cv_auc': auc_cv})
        if auc_cv > best_auc:
            best_auc = auc_cv
            best_alpha = float(alpha)
            best_lambda = lambda_value
            best_pipeline = pipeline

    return {
        'best_model': best_pipeline,
        'best_alpha': best_alpha,
        'best_lambda': best_lambda,
        'best_auc': best_auc,
        'search_results': pd.DataFrame(results).sort_values('cv_auc', ascending=False).reset_index(drop=True),
    }


---
## 5. READ EXCEL FILE
---


In [6]:
df_raw = read_prism_excel(github_xlsx_url)
df_raw.columns = clean_names_simple(df_raw.columns)

print('Rows:', len(df_raw))
print('Columns:', len(df_raw.columns))
print()
print('Column names after cleaning:')
print(list(df_raw.columns))
print()


Could not download the Excel file; using local repo copy instead:
C:\Users\i.Nikesh.Desai\OneDrive - Acentra\Documents\GitHub\prism_repo\DataSets\PRP_1000_full_pretreatment.xlsx
Download error: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Basic Constraints of CA cert not marked critical (_ssl.c:1032)>

Rows: 1000
Columns: 44

Column names after cleaning:
['client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m', 'rx

---
## 6. CHECK REQUIRED COLUMNS
---


In [7]:
required_fields = ['outcome_ed_90d', 'intervention_flag']
require_columns(df_raw, required_fields)


---
## 7. BASIC CLEANUP
---


In [8]:
df = df_raw.copy()

for column in ['index_date', 'intervention_start_date', 'intervention_end_date']:
    if column in df.columns:
        df[column] = safe_as_date(df[column])

df['intervention_flag'] = to_binary(df['intervention_flag'])
df['outcome_ed_90d'] = to_binary(df['outcome_ed_90d'])


---
## 8. DERIVE DATE FEATURES
---


In [9]:
if 'intervention_start_date' in df.columns:
    df['intervention_start_month'] = df['intervention_start_date'].dt.month.astype(float)
    df['intervention_start_wday'] = (((df['intervention_start_date'].dt.dayofweek + 1) % 7) + 1).astype(float)
else:
    df['intervention_start_month'] = np.nan
    df['intervention_start_wday'] = np.nan

if {'index_date', 'intervention_start_date'}.issubset(df.columns):
    df['days_to_intervention_start'] = (df['intervention_start_date'] - df['index_date']).dt.days.astype(float)
else:
    df['days_to_intervention_start'] = np.nan

if {'intervention_start_date', 'intervention_end_date'}.issubset(df.columns):
    df['intervention_duration_calc'] = (df['intervention_end_date'] - df['intervention_start_date']).dt.days.astype(float)
else:
    df['intervention_duration_calc'] = np.nan

if 'intervention_days_active' not in df.columns:
    df['intervention_days_active'] = df['intervention_duration_calc']

# reverse for now
df = df_raw.copy()


---
## 9. SELECT PREDICTORS
---


In [10]:
candidate_predictors_all = [
    'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender',
    'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag',
    'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag',
    'substance_use_flag', 'ckd_flag', 'pregnancy_flag', 'behavioral_health_risk_flag',
    'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag',
    'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m',
    'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m',
    'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
    'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier',
    'intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts',
    'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag',
    'community_referral_flag', 'pharmacy_review_flag', 'engagement_level',
    'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday',
]

candidate_predictors = [column for column in candidate_predictors_all if column in df.columns]
missing_predictors = [column for column in candidate_predictors_all if column not in df.columns]

if missing_predictors:
    print('Predictors not found in dataset:')
    print(missing_predictors)
else:
    print('All candidate predictors are present in dataset.')

model_df = df[['outcome_ed_90d', 'intervention_flag', *candidate_predictors]].copy()
model_df['outcome_ed_90d'] = to_binary(model_df['outcome_ed_90d'])
model_df['intervention_flag'] = to_binary(model_df['intervention_flag'])
model_df = model_df[model_df['outcome_ed_90d'].notna() & model_df['intervention_flag'].notna()].copy()

missing_in_model_df = [column for column in df.columns if column not in model_df.columns]
if missing_in_model_df:
    print('Columns in dataframe but not in model:')
    print(missing_in_model_df)
else:
    print('All dataframe columns are present in model.')
print('Modeling rows after dropping missing outcome/treatment:', len(model_df))
print()


Predictors not found in dataset:
['intervention_type', 'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts', 'avg_call_duration_min', 'max_call_duration_min', 'notes_escalation_flag', 'community_referral_flag', 'pharmacy_review_flag', 'engagement_level', 'days_to_intervention_start', 'intervention_start_month', 'intervention_start_wday']
All dataframe columns are present in model.
Modeling rows after dropping missing outcome/treatment: 1000



---
## 10. DATA TYPE HANDLING
---


In [12]:
model_df.columns

Index(['outcome_ed_90d', 'intervention_flag', 'client_contract',
       'service_region', 'program', 'case_manager_name', 'age', 'gender',
       'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag',
       'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag',
       'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag',
       'pregnancy_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag',
       'housing_instability_flag', 'transportation_barrier_flag',
       'utilities_insecurity_flag', 'pcp_visits_last_6m',
       'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m',
       'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
       'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag',
       'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score',
       'percolator_clinical_score', 'percolator_sdoh_score',
       'current_risk_score', 'risk_tier'],
      dtype='object')

In [13]:
flag_like_cols = [column for column in model_df.columns if column.endswith('_flag')]
for column in flag_like_cols:
    model_df[column] = to_binary(model_df[column])

possible_numeric_cols = [
    'age', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d',
    'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m',
    'rx_count_last_6m', 'med_adherence_pdc', 'percolator_utilization_score',
    'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score',
    'intervention_days_active', 'touches_per_month', 'outreach_attempts', 'successful_contacts',
    'avg_call_duration_min', 'max_call_duration_min', 'days_to_intervention_start',
    'intervention_start_month', 'intervention_start_wday',
]

for column in present_columns(possible_numeric_cols, model_df):
    model_df[column] = pd.to_numeric(model_df[column], errors='coerce')

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if not pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_categorical(model_df[column])

for column in model_df.columns:
    if column in ['outcome_ed_90d', 'intervention_flag']:
        continue
    if pd.api.types.is_numeric_dtype(model_df[column]):
        model_df[column] = impute_numeric(model_df[column])

unique_counts = model_df.apply(lambda column: column.dropna().nunique())
keep_cols = list(unique_counts[unique_counts > 1].index)
model_df = model_df.loc[:, keep_cols].copy().reset_index(drop=True)

print('Final modeling columns:')
print(list(model_df.columns))
print()


Final modeling columns:
['outcome_ed_90d', 'intervention_flag', 'client_contract', 'service_region', 'program', 'case_manager_name', 'age', 'gender', 'dual_eligible', 'county', 'plan_type', 'language', 'living_alone_flag', 'diabetes_flag', 'chf_flag', 'copd_flag', 'asthma_flag', 'depression_flag', 'anxiety_flag', 'substance_use_flag', 'ckd_flag', 'behavioral_health_risk_flag', 'food_insecurity_flag', 'housing_instability_flag', 'transportation_barrier_flag', 'utilities_insecurity_flag', 'pcp_visits_last_6m', 'specialist_visits_last_6m', 'ed_visits_last_30d', 'ed_visits_last_6m', 'admits_last_6m', 'observation_stays_last_6m', 'total_cost_last_6m', 'rx_count_last_6m', 'med_adherence_pdc', 'high_cost_drug_flag', 'opioid_flag', 'polypharmacy_flag', 'percolator_utilization_score', 'percolator_clinical_score', 'percolator_sdoh_score', 'current_risk_score', 'risk_tier']



---
## 11. TRAIN / TEST SPLIT
---


In [14]:
train_df, test_df = split_train_test(model_df, train_fraction=0.70, seed=123)

print('Training rows:', len(train_df))
print('Testing rows:', len(test_df))
print()


Training rows: 700
Testing rows: 300



---
## 12. SEPARATE TREATED / CONTROL
---


In [15]:
train_treated = train_df[train_df['intervention_flag'] == 1].copy()
train_control = train_df[train_df['intervention_flag'] == 0].copy()

print('Training treated rows:', len(train_treated))
print('Training control rows:', len(train_control))
print()

if len(train_treated) < 50:
    raise ValueError('Too few treated rows to train a stable model.')
if len(train_control) < 50:
    raise ValueError('Too few control rows to train a stable model.')


Training treated rows: 272
Training control rows: 428



---
## 13. BUILD MODEL MATRICES
---


In [16]:
feature_cols = [column for column in model_df.columns if column not in ['outcome_ed_90d', 'intervention_flag']]

train_treated_x_df = train_treated[feature_cols].copy()
train_control_x_df = train_control[feature_cols].copy()
test_x_df = test_df[feature_cols].copy()

combined_matrix, split_matrices = make_design_matrix([train_treated_x_df, train_control_x_df, test_x_df])
x_treated, x_control, x_test = split_matrices

y_treated = train_treated['outcome_ed_90d'].astype(float).to_numpy()
y_control = train_control['outcome_ed_90d'].astype(float).to_numpy()


---
## 14. TRAIN XGBOOST MODELS
---


In [ ]:
print('Unique y_treated values:')
print(np.sort(pd.unique(y_treated)))
print()

print('Unique y_control values:')
print(np.sort(pd.unique(y_control)))
print()

if not set(pd.Series(y_treated).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_treated contains values other than 0 and 1.')
if not set(pd.Series(y_control).dropna().unique()).issubset({0.0, 1.0}):
    raise ValueError('y_control contains values other than 0 and 1.')

dtrain_treated = make_dmatrix(x_treated, y_treated)
dtrain_control = make_dmatrix(x_control, y_control)

params = xgb_training_params(
    XGB_GPU_PARAMS,
    {
        'max_depth': 4,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
    eval_metric='logloss',
    seed=123,
)

model_treated = xgb.train(params=params, dtrain=dtrain_treated, num_boost_round=150, verbose_eval=False)
model_control = xgb.train(params=params, dtrain=dtrain_control, num_boost_round=150, verbose_eval=False)
assert_xgb_booster_uses_cuda(model_treated, 'Baseline treated XGBoost model')
assert_xgb_booster_uses_cuda(model_control, 'Baseline control XGBoost model')

print('Models trained successfully with XGBoost GPU params:', XGB_GPU_PARAMS)
print()

test_treated_pos = np.where(test_df['intervention_flag'].to_numpy() == 1)[0]
test_control_pos = np.where(test_df['intervention_flag'].to_numpy() == 0)[0]

pred_treated = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated, 'XGBoost Treated model')

pred_control = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control, 'XGBoost Control model')


---
## XGBOOST CV GRID SEARCH FUNCTION
---


In [18]:
xgb_grid = [
    {'max_depth': max_depth, 'eta': eta, 'min_child_weight': min_child_weight}
    for max_depth, eta, min_child_weight in product([3, 4, 5], [0.03, 0.05, 0.10], [1, 5])
]


---
## TRAIN TREATED MODEL WITH CV GRID SEARCH
---


In [ ]:
xgb_treated_cv = fit_xgb_cv_grid(x_matrix=x_treated, y=y_treated, grid=xgb_grid, nrounds_max=500, nfold=5)
model_treated = xgb_treated_cv['model']
assert_xgb_booster_uses_cuda(model_treated, 'CV-tuned treated XGBoost model')

print('XGBoost Treated best CV AUC:', round(xgb_treated_cv['best_cv_auc'], 4))
print('XGBoost Treated best nrounds:', xgb_treated_cv['best_nrounds'])
print('XGBoost Treated best params:')
print(xgb_treated_cv['best_params'])


---
## TRAIN CONTROL MODEL WITH CV GRID SEARCH
---


In [ ]:
xgb_control_cv = fit_xgb_cv_grid(x_matrix=x_control, y=y_control, grid=xgb_grid, nrounds_max=500, nfold=5)
model_control = xgb_control_cv['model']
assert_xgb_booster_uses_cuda(model_control, 'CV-tuned control XGBoost model')

print('XGBoost Control best CV AUC:', round(xgb_control_cv['best_cv_auc'], 4))
print('XGBoost Control best nrounds:', xgb_control_cv['best_nrounds'])
print('XGBoost Control best params:')
print(xgb_control_cv['best_params'])


---
## TEST AUC FOR CV-TUNED XGBOOST MODELS
---


In [ ]:
pred_treated_cv_xgb = model_treated.predict(make_dmatrix(x_test.iloc[test_treated_pos]))
auc_treated_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_xgb, 'XGBoost Treated CV-tuned test')

pred_control_cv_xgb = model_control.predict(make_dmatrix(x_test.iloc[test_control_pos]))
auc_control_cv_xgb = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_xgb, 'XGBoost Control CV-tuned test')

print('CV-tuned XGBoost models trained successfully.')
print()


---
## OPTIONAL CPU-ONLY GLMNET COMPARISON (SKIPPED BY DEFAULT)
---


In [ ]:
if RUN_CPU_ONLY_COMPARISON_MODELS:
    enet_treated = fit_elastic_net(x_treated, y_treated)

    print('Best treated alpha:', enet_treated['best_alpha'])
    print('Best treated lambda:', enet_treated['best_lambda'])
    print('Best treated CV AUC:', round(enet_treated['best_auc'], 4))
    print()

    enet_control = fit_elastic_net(x_control, y_control)

    print('Best control alpha:', enet_control['best_alpha'])
    print('Best control lambda:', enet_control['best_lambda'])
    print('Best control CV AUC:', round(enet_control['best_auc'], 4))
    print()
else:
    enet_treated = None
    enet_control = None
    print('Skipped GLMNET comparison models because sklearn LogisticRegressionCV trains on CPU.')
    print('Leave RUN_CPU_ONLY_COMPARISON_MODELS = False for GPU-only training runs.')
    print()


---
## OPTIONAL TEST AUC FOR CPU-ONLY GLMNET COMPARISON
---


In [ ]:
if enet_treated is not None and enet_control is not None:
    pred_treated_cv_glmnet = enet_treated['best_model'].predict_proba(x_test.iloc[test_treated_pos])[:, 1]
    auc_treated_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_treated_pos], pred_treated_cv_glmnet, 'GLMNET Treated CV-tuned test')

    pred_control_cv_glmnet = enet_control['best_model'].predict_proba(x_test.iloc[test_control_pos])[:, 1]
    auc_control_cv_glmnet = safe_auc(test_df['outcome_ed_90d'].iloc[test_control_pos], pred_control_cv_glmnet, 'GLMNET Control CV-tuned test')
else:
    pred_treated_cv_glmnet = None
    pred_control_cv_glmnet = None
    auc_treated_cv_glmnet = np.nan
    auc_control_cv_glmnet = np.nan
    print('Skipped GLMNET test AUC because CPU-only GLMNET training was skipped.')
print()


---
## 15. SCORE TEST SET
---


In [ ]:
p_treated = model_treated.predict(make_dmatrix(x_test))
p_control = model_control.predict(make_dmatrix(x_test))

results_test = test_df.copy()
results_test['pred_ed_if_treated'] = p_treated
results_test['pred_ed_if_control'] = p_control
results_test['benefit_score'] = results_test['pred_ed_if_control'] - results_test['pred_ed_if_treated']
results_test['uplift_bad_outcome'] = results_test['pred_ed_if_treated'] - results_test['pred_ed_if_control']
results_test['uplift_decile'] = ntile_desc(results_test['benefit_score'], 10).to_numpy()

print('Top 20 highest-benefit members:')
print(results_test.sort_values('benefit_score', ascending=False)[['outcome_ed_90d', 'intervention_flag', 'pred_ed_if_treated', 'pred_ed_if_control', 'benefit_score', 'uplift_decile']].head(20))
print()


---
## 16. DECILE SUMMARY
---


In [ ]:
decile_summary = (
    results_test
    .groupby('uplift_decile', as_index=False)
    .agg(
        n=('outcome_ed_90d', 'size'),
        avg_benefit_score=('benefit_score', 'mean'),
        observed_ed_rate=('outcome_ed_90d', 'mean'),
        treated_pct=('intervention_flag', 'mean'),
        avg_pred_ed_if_treated=('pred_ed_if_treated', 'mean'),
        avg_pred_ed_if_control=('pred_ed_if_control', 'mean'),
    )
    .sort_values('uplift_decile')
)

print('Decile summary:')
print(decile_summary)
print()


---
## 17. VARIABLE IMPORTANCE
---


In [ ]:
importance_treated = xgb_importance_frame(model_treated)
importance_control = xgb_importance_frame(model_control)

print('Top variables in treated model:')
print(importance_treated.head(20))
print()

print('Top variables in control model:')
print(importance_control.head(20))
print()


---
## 18. SCORE FULL FILE
---


In [ ]:
full_x_df = model_df[feature_cols].copy()
_, [full_matrix_raw] = make_design_matrix([full_x_df])
full_matrix = align_to_columns(full_matrix_raw, combined_matrix.columns)

full_pred_treated = model_treated.predict(make_dmatrix(full_matrix))
full_pred_control = model_control.predict(make_dmatrix(full_matrix))

scored_full = model_df.copy()
scored_full['pred_ed_if_treated'] = full_pred_treated
scored_full['pred_ed_if_control'] = full_pred_control
scored_full['benefit_score'] = scored_full['pred_ed_if_control'] - scored_full['pred_ed_if_treated']
scored_full['uplift_bad_outcome'] = scored_full['pred_ed_if_treated'] - scored_full['pred_ed_if_control']
scored_full['uplift_decile'] = ntile_desc(scored_full['benefit_score'], 10).to_numpy()


---
## 19. WRITE OUTPUTS
---


In [ ]:
scored_full.to_csv(output_path, index=False)
decile_summary.to_csv(summary_path, index=False)

print('Scored full file written to:', output_path, '\n')
print('Decile summary written to:', summary_path, '\n')


---
## 20. INTERPRETATION
---


In [ ]:
print('INTERPRETATION:')
print('- pred_ed_if_treated = predicted probability of ED within 90d if treated')
print('- pred_ed_if_control = predicted probability of ED within 90d if not treated')
print('- benefit_score = pred_ed_if_control - pred_ed_if_treated')
print('- Higher benefit_score means treatment is predicted to reduce ED risk more')
print('- Uplift decile 1 = highest predicted treatment benefit')


---
## DASHBOARD VIEW
---


In [ ]:
dashboard_folder = output_folder


def save_bar_chart(df, x_col, y_col, title, x_label, y_label, path, width=8, height=5):
    fig, ax = plt.subplots(figsize=(width, height))
    ax.bar(df[x_col].astype(str), df[y_col])
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


save_bar_chart(decile_summary, 'uplift_decile', 'avg_benefit_score', 'Average Predicted Intervention Benefit by Uplift Decile', 'Uplift Decile: 1 = Highest Predicted Benefit', 'Average Benefit Score', dashboard_folder / 'dashboard_avg_benefit_by_decile.png')
save_bar_chart(decile_summary, 'uplift_decile', 'observed_ed_rate', 'Observed 90-Day ED Rate by Uplift Decile', 'Uplift Decile', 'Observed ED Rate', dashboard_folder / 'dashboard_observed_ed_rate_by_decile.png')
save_bar_chart(decile_summary, 'uplift_decile', 'treated_pct', 'Current Treatment Penetration by Uplift Decile', 'Uplift Decile', 'Percent Treated', dashboard_folder / 'dashboard_treated_pct_by_decile.png')

decile_long = decile_summary.melt(
    id_vars='uplift_decile',
    value_vars=['avg_pred_ed_if_treated', 'avg_pred_ed_if_control'],
    var_name='scenario',
    value_name='predicted_ed_rate',
)

fig, ax = plt.subplots(figsize=(9, 5))
scenarios = list(decile_long['scenario'].unique())
x_values = np.arange(len(decile_summary))
bar_width = 0.35
for offset, scenario in enumerate(scenarios):
    values = decile_long[decile_long['scenario'] == scenario]['predicted_ed_rate'].to_numpy()
    ax.bar(x_values + (offset - 0.5) * bar_width, values, width=bar_width, label=scenario)
ax.set_xticks(x_values)
ax.set_xticklabels(decile_summary['uplift_decile'].astype(str))
ax.set_title('Predicted ED Risk: Treated vs Control by Decile')
ax.set_xlabel('Uplift Decile')
ax.set_ylabel('Predicted ED Rate')
ax.legend()
fig.tight_layout()
fig.savefig(dashboard_folder / 'dashboard_predicted_treated_vs_control.png', dpi=150)
plt.close(fig)

print('Dashboard charts saved to:', dashboard_folder)


---
## ROI PER DECILE
---


In [ ]:
cost_per_ed_visit = 1200
cost_per_intervention = 250

roi_summary = decile_summary.copy()
roi_summary['expected_ed_rate_reduction'] = roi_summary['avg_benefit_score']
roi_summary['expected_ed_visits_avoided'] = roi_summary['n'] * roi_summary['expected_ed_rate_reduction']
roi_summary['gross_savings'] = roi_summary['expected_ed_visits_avoided'] * cost_per_ed_visit
roi_summary['intervention_cost'] = roi_summary['n'] * cost_per_intervention
roi_summary['net_savings'] = roi_summary['gross_savings'] - roi_summary['intervention_cost']
roi_summary['roi'] = roi_summary['net_savings'] / roi_summary['intervention_cost']

print(roi_summary)
roi_summary.to_csv(dashboard_folder / 'uplift_roi_by_decile.csv', index=False)
save_bar_chart(roi_summary, 'uplift_decile', 'net_savings', 'Estimated Net Savings by Uplift Decile', 'Uplift Decile', 'Estimated Net Savings', dashboard_folder / 'dashboard_roi_net_savings_by_decile.png')
print('ROI summary saved.')


---
## SHAP EXPLANATIONS
---


In [ ]:
shap_treated_importance = shap_importance_frame(model_treated, x_test, 'Treated Model')
shap_control_importance = shap_importance_frame(model_control, x_test, 'Control Model')
shap_importance_combined = pd.concat([shap_treated_importance, shap_control_importance], ignore_index=True)
shap_importance_combined.to_csv(dashboard_folder / 'shap_importance_treated_control_models.csv', index=False)

for shap_df, title, filename in [
    (shap_treated_importance, 'Top SHAP Drivers: Treated Model', 'dashboard_shap_treated_model.png'),
    (shap_control_importance, 'Top SHAP Drivers: Control Model', 'dashboard_shap_control_model.png'),
]:
    top = shap_df.nlargest(20, 'mean_abs_shap').sort_values('mean_abs_shap')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top['feature'], top['mean_abs_shap'])
    ax.set_title(title)
    ax.set_xlabel('Mean Absolute SHAP Contribution')
    ax.set_ylabel('Feature')
    fig.tight_layout()
    fig.savefig(dashboard_folder / filename, dpi=150)
    plt.close(fig)

print('SHAP outputs saved.')
print('Files currently in output folder:')
print('\n'.join(str(path) for path in sorted(output_folder.iterdir())))
